In [ ]:
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import numpy as np
from xgboost import XGBRegressor
import json
import os
import tempfile
from tqdm import tqdm
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import subprocess


In [ ]:
import sys
from pathlib import Path
workspace_root = Path().cwd().parent.parent


print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

from util import evaluate
from community_contributions.yeechern.modal_package import app, run_embedding, train_evaluate_xgboost

In [ ]:
env_path = workspace_root.parent / ".env"
load_dotenv(override=True)

print("HF_TOKEN loaded!" if os.environ.get("HF_TOKEN") else "Failed to load HF_TOKEN")
print("MODAL_TOKEN_ID loaded!" if os.environ.get("MODAL_TOKEN_ID") else "Failed to load MODAL_TOKEN_ID")
print("MODAL_TOKEN_SECRET loaded!" if os.environ.get("MODAL_TOKEN_SECRET") else "Failed to load MODAL_TOKEN_SECRET")

In [ ]:
os.environ["MODAL_IMAGE_BUILDER_VERSION"] = "2025.06"

## Embedding on Modal.com

In [ ]:
DATASET_NAME = "ed-donner/items_prompts_full"

# get embedded dataset
embedding_model = "sentence-transformers/all-MiniLM-L12-v2"

with app.run():
    modal_volume_name, file_name = run_embedding.remote(DATASET_NAME, embedding_model = embedding_model, batch_size=512)

In [ ]:
local_data_dir = Path("./data")

subprocess.run([
    "modal", "volume", "get",
    modal_volume_name, file_name, str(local_data_dir)
], check=True)

In [ ]:
dataset = load_from_disk(local_data_dir/ file_name)
dataset, dataset['train']["embedding"][0].shape if type(dataset['train']["embedding"][0]) == np.ndarray else len(dataset['train']["embedding"][0])

### Preprocessing

In [ ]:
# price string to float
dataset = dataset.with_format("numpy")
dataset = dataset.map(lambda x : {'completion': float(x['completion'])}, batched=False)

In [ ]:
# price normalization
scaler = StandardScaler()
scaler.fit(np.array(dataset['train']['completion']).reshape(-1, 1))

def normalize(batch):
    arr = np.asarray(batch["completion"], dtype=np.float64).reshape(-1, 1)
    return {"completion": scaler.transform(arr).flatten().tolist()}

train_dataset = dataset['train'].map(normalize, batched=True)
val_dataset = dataset['val'].map(normalize, batched=True)

## Train XGBoost model with embeddings

In [ ]:
X_train = np.asarray(train_dataset['embedding'], dtype=np.float32)
y_train = train_dataset['completion']

X_val = np.asarray(val_dataset['embedding'], dtype=np.float32)
y_val = val_dataset['completion']

In [ ]:
X_train

In [ ]:
model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

Training stop at 1301 and rmse is 0.63037.

In [ ]:
def model_predict(item):
    inputs = item['embedding']
    
    prediction = model.predict(np.array([inputs], dtype=np.float32))
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

Error: 56.52 ± 9.51, MSE: 7,906, r2: 64.0%

### Try Remove Outliers from training data

In [ ]:
q3 = np.percentile(train_dataset['completion'], 75)
q1 = np.percentile(train_dataset['completion'], 25)
cap = q3 + 1.5 * (q3 - q1)

train_dataset_capped = train_dataset.filter(lambda x: x['completion'] <= cap)

In [ ]:
X_train = np.asarray(train_dataset_capped['embedding'], dtype=np.float32)
y_train = train_dataset_capped['completion']

model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

Training stop at 1572 and rmse is 0.76978.

In [ ]:
def model_predict(item):
    inputs = item['embedding']
    
    prediction = model.predict(np.array([inputs], dtype=np.float32))
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

Error: 62.25 ± 13.19, MSE: 12,928, r2: 41.2%

Disappointingly, the model performance is not improved after removing outliers.

## Try PCA to reduce the dimension of embeddings

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)

X_train = np.asarray(train_dataset['embedding'], dtype=np.float32)
y_train = train_dataset['completion']

X_val = np.asarray(val_dataset['embedding'], dtype=np.float32)
y_val = val_dataset['completion']

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)

model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

Training stop at 1016 and rmse is 0.67517.

In [ ]:
def model_predict(item):
    inputs = item['embedding']
    inputs = pca.transform(np.array([inputs], dtype=np.float32))
    
    prediction = model.predict(inputs)
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

Error: 56.74 ± 8.85, MSE: 7,294, r2: 66.8%

Seem like PCA didn't improve the model performance either.

## Try using larger embedding model

In [ ]:
embedding_model = 'BAAI/bge-large-en-v1.5'

with app.run():
    modal_volume_name, file_name = run_embedding.remote(DATASET_NAME, embedding_model = embedding_model, batch_size=256)

In [ ]:
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import numpy as np
from xgboost import XGBRegressor
import json
import os
import tempfile
from tqdm import tqdm
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import subprocess

import sys
from pathlib import Path
workspace_root = Path().cwd().parent.parent


print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

from util import evaluate
from community_contributions.yeechern.modal_package import app, run_embedding, train_evaluate_xgboost
env_path = workspace_root.parent / ".env"
load_dotenv(override=True)

print("HF_TOKEN loaded!" if os.environ.get("HF_TOKEN") else "Failed to load HF_TOKEN")
print("MODAL_TOKEN_ID loaded!" if os.environ.get("MODAL_TOKEN_ID") else "Failed to load MODAL_TOKEN_ID")
print("MODAL_TOKEN_SECRET loaded!" if os.environ.get("MODAL_TOKEN_SECRET") else "Failed to load MODAL_TOKEN_SECRET")
os.environ["MODAL_IMAGE_BUILDER_VERSION"] = "2025.06"

In [ ]:
modal_volume_name = "ed-donner-price-prediction"
DATASET_NAME = "ed-donner/items_prompts_full"
embedding_model = 'BAAI/bge-large-en-v1.5'

file_name = DATASET_NAME.split("/")[-1] + "_embedded_" + embedding_model.split("/")[-1]

with app.run():
    results = train_evaluate_xgboost.remote(file_name)
    

In [ ]:
import pandas as pd
from util import Tester

results = pd.DataFrame(results)

In [ ]:
results.info()

In [ ]:
tester = Tester(model_predict,None,None,len(results))
tester.titles = results['title'].tolist()
tester.guesses = results['guess'].tolist()
tester.truths = results['truth'].tolist()
tester.errors = results['error'].tolist()
tester.colors = results['color'].tolist()

tester.report()

Error: 56.52 ± 9.51, MSE: 7,906, r2: 64.0%

## Conclusion

- Remove outliers and PCA didn't improve the model performance.
- BAAI/bge-large-en-v1.5 embedding didn't superior to sentence-transformers/all-MiniLM-L12-v2 embedding (I have also tried sentence-transformers/all-MiniLM-L6-v2, which is smaller than L12-v2, but the performance is slightly worse than L12-v2).

Results table:

| Approach | RMSE | MSE | R2 |
| --- | --- | --- | --- |
| all-MiniLM-L12-v2 | 56.52 | 7,906 | 64.0% |
| all-MiniLM-L12-v2 + outliers removed | 62.25 | 12,928 | 41.2% |
| all-MiniLM-L12-v2 + PCA | 56.74 | 7,294 | 66.8% |
| bge-large-en-v1.5 | 56.52 | 7,906 | 64.0% |


## Future Work

- Try hyperparameter tuning with BAAI/bge-large-en-v1.5, since BAAI/bge-large-en-v1.5 embedded dataset requires more memory, modal.com is my only option to train xgboost, and I have not tried hyperparameter tuning on modal.com yet.